# Ensembles for in-domain and cross-domain offensive language detection

Base models from Assignment 4: fBERT, HateBERT, Logistic Regression and CNN.
- **In-domain:** train on `olid-train-small.csv`, test on `olid-test.csv`
- **Cross-domain:** train on `hasoc-train.csv`, test on `olid-test.csv`

5-fold cross-validation on each training set gives out-of-fold predictions, which are used to select the base models and to train the stacking meta-model. The test set is only used for the final evaluation.

Fold predictions are cached in `results/kfold/`, test predictions in `results/test_*.csv`, fine-tuned transformers in `models/`.

**Setup:**
```bash
python3.13 -m venv venv-a4
source venv-a4/bin/activate
pip install torch transformers accelerate tensorflow scikit-learn pandas matplotlib ipykernel
```

In [1]:
import os
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import tensorflow as tf
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from tensorflow.keras import Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    TextVectorization,
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense,
    Dropout
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedGroupKFold, PredefinedSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import precision_recall_fscore_support

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print("Device:", DEVICE)

/Users/justus/Desktop/projects/subjective-assignment-3/venv-a4/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: mps


In [2]:
SEED = 42
MAX_SEQ_LENGTH = 512
EVAL_BATCH_SIZE = 100
N_FOLDS = 5
CNN_SEED = 0

# simpletransformers ClassificationArgs defaults
TRAINING_ARGS = dict(
    num_train_epochs=1,
    learning_rate=4e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    warmup_steps=0.06,
    lr_scheduler_type="linear",
    weight_decay=0.0,
    adam_epsilon=1e-8,
    adam_beta1=0.9,
    adam_beta2=0.999,
    max_grad_norm=1.0,
    optim="adamw_torch",
    seed=SEED,
)

TRANSFORMERS = {"fbert": "diptanu/fBERT", "hatebert": "GroNLP/hateBERT"}
MODELS = ["fbert", "hatebert", "lr", "cnn"]

os.makedirs("results/kfold", exist_ok=True)
os.makedirs("figures", exist_ok=True)

## Data

In [3]:
olid_train = pd.read_csv("data/olid-train-small.csv")
olid_test = pd.read_csv("data/olid-test.csv")
hasoc_train = pd.read_csv("data/hasoc-train.csv")

print("OLID train:", olid_train.shape)
print("OLID test:", olid_test.shape)
print("HASOC train:", hasoc_train.shape)

OLID train: (5852, 3)
OLID test: (860, 3)
HASOC train: (5852, 3)


In [4]:
# Grouping by text keeps duplicate messages in the same fold
def assign_folds(df):
    df = df.copy()
    folds = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fold, (_, held_out) in enumerate(folds.split(df["text"], df["labels"], groups=df["text"])):
        df.loc[df.index[held_out], "fold"] = fold
    df["fold"] = df["fold"].astype(int)
    return df


# Setups are named after their training set: olid = in-domain, hasoc = cross-domain
TRAIN_SETS = {"olid": assign_folds(olid_train), "hasoc": assign_folds(hasoc_train)}

for setup, df in TRAIN_SETS.items():
    print(setup, df.groupby("fold")["labels"].agg(["size", "mean"]).round(3).to_dict("index"))

olid {0: {'size': 1171, 'mean': 0.387}, 1: {'size': 1170, 'mean': 0.386}, 2: {'size': 1170, 'mean': 0.386}, 3: {'size': 1171, 'mean': 0.386}, 4: {'size': 1170, 'mean': 0.386}}
hasoc {0: {'size': 1170, 'mean': 0.386}, 1: {'size': 1174, 'mean': 0.387}, 2: {'size': 1170, 'mean': 0.386}, 3: {'size': 1170, 'mean': 0.386}, 4: {'size': 1168, 'mean': 0.386}}


## Base models

### Transformers (fBERT, HateBERT)

In [5]:
class TextDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer):
        self.encodings = tokenizer(list(df["text"]), truncation=True, max_length=MAX_SEQ_LENGTH)
        self.labels = list(df["labels"])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.encodings.items()}
        item["labels"] = self.labels[i]
        return item


def train_transformer(model_name, train_df):
    set_seed(SEED)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    args = TrainingArguments(
        output_dir="models/tmp",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
        dataloader_pin_memory=False,
        **TRAINING_ARGS,
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=TextDataset(train_df, tokenizer),
        data_collator=DataCollatorWithPadding(tokenizer),
    )
    trainer.train()
    return trainer.model, tokenizer


@torch.no_grad()
def predict_transformer(model, tokenizer, test_df):
    model = model.to(DEVICE).eval()

    probabilities = []
    for i in range(0, len(test_df), EVAL_BATCH_SIZE):
        batch = tokenizer(
            list(test_df["text"].iloc[i:i + EVAL_BATCH_SIZE]),
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
            padding=True,
            return_tensors="pt"
        ).to(DEVICE)
        probabilities.append(model(**batch).logits.softmax(dim=-1)[:, 1].cpu().numpy())

    return np.concatenate(probabilities)


def transformer_probabilities(model_name, train_df, test_df):
    model, tokenizer = train_transformer(model_name, train_df)
    probabilities = predict_transformer(model, tokenizer, test_df)
    del model
    if DEVICE == "mps":
        torch.mps.empty_cache()
    return probabilities

### Logistic Regression

In [6]:
def lr_probabilities(train_df, test_df):
    model = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ])
    model.fit(train_df["text"], train_df["labels"])
    return model.predict_proba(test_df["text"])[:, 1]

### CNN

In [7]:
def cnn_probabilities(train_df, test_df, seed=CNN_SEED):
    tf.keras.utils.set_random_seed(seed)

    train_part, dev_part = train_test_split(
        train_df, test_size=0.1, stratify=train_df["labels"], random_state=seed
    )

    X_train = tf.constant(train_part["text"].astype(str).tolist())
    y_train = train_part["labels"].values
    X_dev = tf.constant(dev_part["text"].astype(str).tolist())
    y_dev = dev_part["labels"].values
    X_test = tf.constant(test_df["text"].astype(str).tolist())

    vectorizer = TextVectorization(
        max_tokens=20000,
        output_sequence_length=100
    )
    vectorizer.adapt(X_train)

    model = Sequential([
        vectorizer,
        Embedding(20000, 128),
        Conv1D(128, 5, activation="relu"),
        GlobalMaxPooling1D(),
        Dropout(0.5),
        Dense(64, activation="relu"),
        Dropout(0.5),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stopping = EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    )

    model.fit(
        X_train,
        y_train,
        validation_data=(X_dev, y_dev),
        epochs=20,
        batch_size=32,
        callbacks=[early_stopping],
        verbose=0
    )

    return model.predict(X_test, verbose=0).flatten()

In [8]:
def probabilities(model, train_df, test_df):
    if model in TRANSFORMERS:
        return transformer_probabilities(TRANSFORMERS[model], train_df, test_df)
    if model == "lr":
        return lr_probabilities(train_df, test_df)
    return cnn_probabilities(train_df, test_df)

## K-fold cross-validation

In [9]:
def run_kfold(model, setup):
    train_df = TRAIN_SETS[setup]
    folds = []

    for fold in range(N_FOLDS):
        path = f"results/kfold/{model}_{setup}_fold{fold}.csv"

        if os.path.exists(path):
            results = pd.read_csv(path)
        else:
            held_out = train_df["fold"] == fold
            results = train_df.loc[held_out, ["id", "text", "labels", "fold"]].copy()
            results["probability"] = probabilities(model, train_df[~held_out], results)
            results["prediction"] = (results["probability"] >= 0.5).astype(int)
            results.to_csv(path, index=False)
            print(f"{model} {setup} fold {fold} saved to {path}")

        folds.append(results)

    return pd.concat(folds).set_index("id").loc[train_df["id"]].reset_index()


oof = {(model, setup): run_kfold(model, setup) for model in ["lr", "cnn", "fbert", "hatebert"] for setup in TRAIN_SETS}

## Base model selection

All numbers in this section are computed on the out-of-fold predictions of the training sets.

In [10]:
def scores(labels, predictions):
    _, _, f1, _ = precision_recall_fscore_support(labels, predictions, labels=[0, 1], zero_division=0)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels, predictions, average="macro", zero_division=0
    )
    return {
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "NOT F1": f1[0],
        "OFF F1": f1[1],
        "Predicted OFF": np.mean(predictions),
    }


oof_scores = pd.DataFrame([
    {"Setup": setup, "Model": model, **scores(oof[(model, setup)]["labels"], oof[(model, setup)]["prediction"])}
    for setup in TRAIN_SETS
    for model in MODELS
]).set_index(["Setup", "Model"])

oof_scores.round(3)

Macro Precision  Macro Recall  Macro F1  NOT F1  OFF F1  \
Setup Model                                                               
olid  fbert               0.802         0.804     0.803   0.847   0.759   
      hatebert            0.776         0.771     0.773   0.828   0.718   
      lr                  0.730         0.656     0.658   0.799   0.517   
      cnn                 0.721         0.671     0.676   0.797   0.555   
hasoc fbert               0.695         0.656     0.660   0.782   0.538   
      hatebert            0.678         0.648     0.652   0.771   0.533   
      lr                  0.649         0.611     0.609   0.759   0.458   
      cnn                 0.627         0.613     0.615   0.734   0.496   

                Predicted OFF  
Setup Model                    
olid  fbert             0.391  
      hatebert          0.370  
      lr                0.202  
      cnn               0.240  
hasoc fbert             0.254  
      hatebert          0.272  
      lr                0.229  
      cnn               0.304

In [11]:
correlations = {
    setup: pd.DataFrame(
        [[np.corrcoef(oof[(a, setup)]["prediction"], oof[(b, setup)]["prediction"])[0, 1] for b in MODELS] for a in MODELS],
        index=MODELS,
        columns=MODELS
    )
    for setup in TRAIN_SETS
}

fig, axes = plt.subplots(1, 2, figsize=(10, 4.2))

for ax, setup in zip(axes, TRAIN_SETS):
    matrix = correlations[setup].values
    image = ax.imshow(matrix, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(MODELS)), MODELS)
    ax.set_yticks(range(len(MODELS)), MODELS)
    for i, j in itertools.product(range(len(MODELS)), repeat=2):
        ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", color="white" if matrix[i, j] > 0.6 else "black")
    ax.set_title(setup)

fig.colorbar(image, ax=axes, shrink=0.8, label="Pearson's r")
plt.savefig("figures/a5_oof_correlations.pdf", bbox_inches="tight")
plt.show()

pd.concat(correlations, names=["Setup"]).round(3)

/var/folders/fg/rxrmf_6n02j12fr7tjtxshb00000gn/T/ipykernel_15547/29557738.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


fbert  hatebert     lr    cnn
Setup                                        
olid  fbert     1.000     0.784  0.502  0.512
      hatebert  0.784     1.000  0.464  0.498
      lr        0.502     0.464  1.000  0.617
      cnn       0.512     0.498  0.617  1.000
hasoc fbert     1.000     0.710  0.494  0.445
      hatebert  0.710     1.000  0.492  0.463
      lr        0.494     0.492  1.000  0.617
      cnn       0.445     0.463  0.617  1.000

In [12]:
rows = []
for setup in TRAIN_SETS:
    labels = oof[("lr", setup)]["labels"].values
    for a, b in itertools.combinations(MODELS, 2):
        correct_a = oof[(a, setup)]["prediction"].values == labels
        correct_b = oof[(b, setup)]["prediction"].values == labels
        rows.append({
            "Setup": setup,
            "Pair": f"{a} - {b}",
            "A right, B wrong": (correct_a & ~correct_b).sum(),
            "B right, A wrong": (correct_b & ~correct_a).sum(),
            "Both wrong": (~correct_a & ~correct_b).sum(),
        })

pd.DataFrame(rows).set_index(["Setup", "Pair"])

A right, B wrong  B right, A wrong  Both wrong
Setup Pair                                                            
olid  fbert - hatebert               376               222         873
      fbert - lr                     982               414         681
      fbert - cnn                    943               404         691
      hatebert - lr                  917               503         746
      hatebert - cnn                 857               472         777
      lr - cnn                       376               405        1258
hasoc fbert - hatebert               364               296        1436
      fbert - lr                     654               435        1297
      fbert - cnn                    812               506        1226
      hatebert - lr                  638               487        1313
      hatebert - cnn                 766               528        1272
      lr - cnn                       498               411        1540

In [13]:
rows = []
for setup in TRAIN_SETS:
    labels = oof[("lr", setup)]["labels"].values
    disagree = oof[("fbert", setup)]["prediction"].values != oof[("hatebert", setup)]["prediction"].values
    for model in MODELS:
        rows.append({
            "Setup": setup,
            "Model": model,
            "Transformers disagree": disagree.sum(),
            "Correct where they disagree": (oof[(model, setup)]["prediction"].values[disagree] == labels[disagree]).mean(),
        })

pd.DataFrame(rows).set_index(["Setup", "Model"]).round(3)

Transformers disagree  Correct where they disagree
Setup Model                                                       
olid  fbert                       598                        0.629
      hatebert                    598                        0.371
      lr                          598                        0.560
      cnn                         598                        0.535
hasoc fbert                       660                        0.552
      hatebert                    660                        0.448
      lr                          660                        0.509
      cnn                         660                        0.518

In [14]:
rows = []
for setup in TRAIN_SETS:
    labels = oof[("lr", setup)]["labels"].values
    best_single = oof_scores.loc[setup, "Macro F1"].max()
    for combo in itertools.combinations(MODELS, 3):
        votes = np.stack([oof[(model, setup)]["prediction"].values for model in combo])
        hard_f1 = scores(labels, (votes.sum(axis=0) >= 2).astype(int))["Macro F1"]
        rows.append({
            "Setup": setup,
            "Combination": " + ".join(combo),
            "Hard voting Macro F1": hard_f1,
            "Difference to best single model": hard_f1 - best_single,
            "Oracle accuracy": (votes == labels).any(axis=0).mean(),
        })

pd.DataFrame(rows).set_index(["Setup", "Combination"]).round(3)

Hard voting Macro F1  \
Setup Combination                                    
olid  fbert + hatebert + lr                  0.791   
      fbert + hatebert + cnn                 0.789   
      fbert + lr + cnn                       0.715   
      hatebert + lr + cnn                    0.703   
hasoc fbert + hatebert + lr                  0.654   
      fbert + hatebert + cnn                 0.658   
      fbert + lr + cnn                       0.630   
      hatebert + lr + cnn                    0.630   

                              Difference to best single model  Oracle accuracy  
Setup Combination                                                               
olid  fbert + hatebert + lr                            -0.012            0.901  
      fbert + hatebert + cnn                           -0.014            0.898  
      fbert + lr + cnn                                 -0.088            0.898  
      hatebert + lr + cnn                              -0.101            0.890  
hasoc fbert + hatebert + lr                            -0.006            0.805  
      fbert + hatebert + cnn                           -0.003            0.814  
      fbert + lr + cnn                                 -0.031            0.813  
      hatebert + lr + cnn                              -0.030            0.808

## Test predictions

Selected base models: fBERT, HateBERT and Logistic Regression. Each is trained on the full training set and predicts `olid-test.csv`.

In [15]:
SELECTED = ["fbert", "hatebert", "lr"]

MODEL_DIRS = {"olid": "in-domain", "hasoc": "cross-domain"}


def full_transformer_probabilities(model, setup):
    model_dir = f"models/{model}-{MODEL_DIRS[setup]}"

    if os.path.exists(os.path.join(model_dir, "config.json")):
        print(f"{model_dir} exists, skipping training")
        tokenizer = AutoTokenizer.from_pretrained(model_dir)
        classifier = AutoModelForSequenceClassification.from_pretrained(model_dir)
    else:
        classifier, tokenizer = train_transformer(TRANSFORMERS[model], TRAIN_SETS[setup])
        classifier.save_pretrained(model_dir)
        tokenizer.save_pretrained(model_dir)

    return predict_transformer(classifier, tokenizer, olid_test)


def run_test(model, setup):
    path = f"results/test_{model}_{setup}.csv"

    if os.path.exists(path):
        print(f"{path} exists, loading saved predictions")
        return pd.read_csv(path)

    results = olid_test[["id", "text", "labels"]].copy()
    if model in TRANSFORMERS:
        results["probability"] = full_transformer_probabilities(model, setup)
    else:
        results["probability"] = probabilities(model, TRAIN_SETS[setup], olid_test)
    results["prediction"] = (results["probability"] >= 0.5).astype(int)
    results.to_csv(path, index=False)
    return results


test = {}
for model in SELECTED:
    for setup in TRAIN_SETS:
        test[(model, setup)] = run_test(model, setup)

results/test_fbert_olid.csv exists, loading saved predictions
results/test_fbert_hasoc.csv exists, loading saved predictions
results/test_hatebert_olid.csv exists, loading saved predictions
results/test_hatebert_hasoc.csv exists, loading saved predictions
results/test_lr_olid.csv exists, loading saved predictions
results/test_lr_hasoc.csv exists, loading saved predictions


## Hard majority voting

In [16]:
ensembles = {}
for setup in TRAIN_SETS:
    votes = np.stack([test[(model, setup)]["prediction"].values for model in SELECTED])
    ensembles[("hard voting", setup)] = (votes.sum(axis=0) >= 2).astype(int)

In [17]:
def evaluate(predictions):
    labels = olid_test["labels"]

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        labels=[0, 1],
        zero_division=0
    )

    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "NOT Precision": precision[0],
        "NOT Recall": recall[0],
        "NOT F1": f1[0],
        "OFF Precision": precision[1],
        "OFF Recall": recall[1],
        "OFF F1": f1[1],
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
    }


def results_table():
    rows = []
    for setup in TRAIN_SETS:
        for model in SELECTED:
            rows.append({"Setup": setup, "Model": model, **evaluate(test[(model, setup)]["prediction"])})
        for (ensemble, ensemble_setup), predictions in ensembles.items():
            if ensemble_setup == setup:
                rows.append({"Setup": setup, "Model": ensemble, **evaluate(predictions)})
    return pd.DataFrame(rows).set_index(["Setup", "Model"]).round(3)


results_table()

NOT Precision  NOT Recall  NOT F1  OFF Precision  \
Setup Model                                                           
olid  fbert                0.887       0.897   0.892          0.725   
      hatebert             0.885       0.918   0.901          0.765   
      lr                   0.802       0.948   0.869          0.748   
      hard voting          0.870       0.932   0.900          0.786   
hasoc fbert                0.866       0.966   0.913          0.875   
      hatebert             0.836       0.952   0.890          0.805   
      lr                   0.780       0.774   0.777          0.429   
      hard voting          0.842       0.955   0.895          0.822   

                   OFF Recall  OFF F1  Macro Precision  Macro Recall  Macro F1  
Setup Model                                                                     
olid  fbert             0.704   0.715            0.806         0.800     0.803  
      hatebert          0.692   0.726            0.825         0.805     0.814  
      lr                0.396   0.518            0.775         0.672     0.693  
      hard voting       0.642   0.706            0.828         0.787     0.803  
hasoc fbert             0.612   0.721            0.870         0.789     0.817  
      hatebert          0.517   0.629            0.820         0.734     0.760  
      lr                0.438   0.433            0.605         0.606     0.605  
      hard voting       0.538   0.650            0.832         0.746     0.772

## Soft majority voting

In [18]:
for setup in TRAIN_SETS:
    mean_probability = np.mean([test[(model, setup)]["probability"].values for model in SELECTED], axis=0)
    ensembles[("soft voting", setup)] = (mean_probability >= 0.5).astype(int)

results_table()

NOT Precision  NOT Recall  NOT F1  OFF Precision  \
Setup Model                                                           
olid  fbert                0.887       0.897   0.892          0.725   
      hatebert             0.885       0.918   0.901          0.765   
      lr                   0.802       0.948   0.869          0.748   
      hard voting          0.870       0.932   0.900          0.786   
      soft voting          0.877       0.924   0.900          0.773   
hasoc fbert                0.866       0.966   0.913          0.875   
      hatebert             0.836       0.952   0.890          0.805   
      lr                   0.780       0.774   0.777          0.429   
      hard voting          0.842       0.955   0.895          0.822   
      soft voting          0.848       0.953   0.897          0.822   

                   OFF Recall  OFF F1  Macro Precision  Macro Recall  Macro F1  
Setup Model                                                                     
olid  fbert             0.704   0.715            0.806         0.800     0.803  
      hatebert          0.692   0.726            0.825         0.805     0.814  
      lr                0.396   0.518            0.775         0.672     0.693  
      hard voting       0.642   0.706            0.828         0.787     0.803  
      soft voting       0.667   0.716            0.825         0.795     0.808  
hasoc fbert             0.612   0.721            0.870         0.789     0.817  
      hatebert          0.517   0.629            0.820         0.734     0.760  
      lr                0.438   0.433            0.605         0.606     0.605  
      hard voting       0.538   0.650            0.832         0.746     0.772  
      soft voting       0.558   0.665            0.835         0.756     0.781

## Stacking

In [1]:
PLACEHOLDERS = r"@USER|\bURL\b"


def extra_features(df):
    text = df["text"].str.replace(PLACEHOLDERS, "", regex=True).str.strip()
    return pd.DataFrame({
        "tokens": text.str.split().str.len(),
        "characters": text.str.len(),
        "uppercase share": text.str.count(r"[A-Z]") / text.str.len().clip(lower=1),
        "exclamation marks": text.str.count("!"),
    })


def meta_inputs(predictions, setup):
    probabilities = pd.DataFrame({model: predictions[(model, setup)]["probability"].values for model in SELECTED})
    return pd.concat([probabilities, extra_features(predictions[(SELECTED[0], setup)])], axis=1)

In [20]:
META_MODELS = {
    "gradient boosting": GradientBoostingClassifier(random_state=SEED),
    "logistic regression": Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=SEED))
    ]),
}

rows = []
for setup in TRAIN_SETS:
    X = meta_inputs(oof, setup)
    y = oof[("lr", setup)]["labels"]
    folds = PredefinedSplit(oof[("lr", setup)]["fold"])
    for name, meta_model in META_MODELS.items():
        f1 = cross_val_score(meta_model, X, y, cv=folds, scoring="f1_macro")
        rows.append({"Setup": setup, "Meta-model": name, "Macro F1": f1.mean(), "Std": f1.std()})

meta_selection = pd.DataFrame(rows).set_index(["Setup", "Meta-model"])
meta_selection.round(3)

Macro F1    Std
Setup Meta-model                          
olid  gradient boosting       0.799  0.005
      logistic regression     0.802  0.005
hasoc gradient boosting       0.650  0.010
      logistic regression     0.662  0.017

In [21]:
for setup in TRAIN_SETS:
    best = meta_selection.loc[setup, "Macro F1"].idxmax()
    meta_model = clone(META_MODELS[best]).fit(meta_inputs(oof, setup), oof[("lr", setup)]["labels"])
    ensembles[("stacking", setup)] = meta_model.predict(meta_inputs(test, setup))
    print(f"{setup}: meta-model = {best}")

results_table()

olid: meta-model = logistic regression
hasoc: meta-model = logistic regression


NOT Precision  NOT Recall  NOT F1  OFF Precision  \
Setup Model                                                           
olid  fbert                0.887       0.897   0.892          0.725   
      hatebert             0.885       0.918   0.901          0.765   
      lr                   0.802       0.948   0.869          0.748   
      hard voting          0.870       0.932   0.900          0.786   
      soft voting          0.877       0.924   0.900          0.773   
      stacking             0.878       0.916   0.897          0.756   
hasoc fbert                0.866       0.966   0.913          0.875   
      hatebert             0.836       0.952   0.890          0.805   
      lr                   0.780       0.774   0.777          0.429   
      hard voting          0.842       0.955   0.895          0.822   
      soft voting          0.848       0.953   0.897          0.822   
      stacking             0.857       0.948   0.900          0.816   

                   OFF Recall  OFF F1  Macro Precision  Macro Recall  Macro F1  
Setup Model                                                                     
olid  fbert             0.704   0.715            0.806         0.800     0.803  
      hatebert          0.692   0.726            0.825         0.805     0.814  
      lr                0.396   0.518            0.775         0.672     0.693  
      hard voting       0.642   0.706            0.828         0.787     0.803  
      soft voting       0.667   0.716            0.825         0.795     0.808  
      stacking          0.671   0.711            0.817         0.793     0.804  
hasoc fbert             0.612   0.721            0.870         0.789     0.817  
      hatebert          0.517   0.629            0.820         0.734     0.760  
      lr                0.438   0.433            0.605         0.606     0.605  
      hard voting       0.538   0.650            0.832         0.746     0.772  
      soft voting       0.558   0.665            0.835         0.756     0.781  
      stacking          0.592   0.686            0.837         0.770     0.793